In [5]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.output_parsers import StrOutputParser
import os


llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    api_key = os.environ['GEMINI_API_KEY'],
    temperature=0
    )


prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder(variable_name="history"), # to insert some message inside prompt template we use MessagesPlaceholder -> we insert
    ("human", "{input}")
])

chain = prompt | llm | StrOutputParser()

store = {}

def get_session_history(session_id):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory() # python memory -> it creates new sesion history and stores in python memory -> after creating it stores history
    return store[session_id]


chain_with_memory = RunnableWithMessageHistory( # runs the chain with message history(memory)
    chain,
    get_session_history,  # runnable will call the function(while calling the function it will pass 1 argument session id which returns chat history)
    input_messages_key="input",  # {"input": "My name is Alice."}, is stored in input
    history_messages_key="history" # we get this from get_session_history the history returned by the function is stored in "history"
)
# now this all 3 will go to chain

user1 = chain_with_memory.invoke(
    {"input": "My name is Alice."},
    config={"configurable": {"session_id": "user_1"}}  # we are passing input and session id(to fetch session history)
)
user2 = chain_with_memory.invoke(
    {"input": "What is my name ?"},
    config={"configurable": {"session_id": "user_1"}}
)
# it connects LCEL and session history
# it runs the chain with memory
print(user1)

c:\Users\LENOVO\AppData\Local\Programs\Python\Python314\Lib\site-packages\IPython\core\interactiveshell.py:3701: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


Hello Alice! It's nice to meet you. How can I help you today?


In [6]:
print(user2)

Your name is Alice. How can I help you today, Alice?


In [15]:
print(prompt[0])
print(prompt[1])
print(prompt[2])

prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a helpful assistant.') additional_kwargs={}
variable_name='history'
prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}') additional_kwargs={}


In [16]:
print(store)

{'user_1': InMemoryChatMessageHistory(messages=[HumanMessage(content='My name is Alice.', additional_kwargs={}, response_metadata={}), AIMessage(content="Hello Alice! It's nice to meet you. How can I help you today?", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='What is my name ?', additional_kwargs={}, response_metadata={}), AIMessage(content='Your name is Alice. How can I help you today, Alice?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])])}


In [17]:
print(store['user_1'])

Human: My name is Alice.
AI: Hello Alice! It's nice to meet you. How can I help you today?
Human: What is my name ?
AI: Your name is Alice. How can I help you today, Alice?


In [18]:
print(store['user_2'])

KeyError: 'user_2'

In [ ]:
# create 3 sessions
# print the store and check what information is stored inside that

# invoke the chain
# 2nd use diffferent senssion id and check whether it is returning your name or not

# check all possible scenarios

# task 2
# create 1 mini project cmd project(in terminal, not ui, use .py file)
# use runnablechathistory

In [ ]:
# pydantic model
# how to create tools and agents
# how to connect all agents by using langgraph